# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kifayat-Sayed/flyrank-ml-internship1/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

### Objective

The objective is to construct a reproducible feature vector for the observed content-decline classification task.

The target is defined as:

- `1`: `trend_direction == "down"`.
- `0`: all other observed trend categories.

The feature vector follows the same exclusion rules used in the final capstone.

**Feature-selection principles**

1. Exclude content and client identifiers from predictive features.
2. Exclude `trend_direction` and `trend_pct`, because they directly define or describe the target.
3. Exclude the recent 30-day and preceding 30-day comparison fields used to derive the trend label.
4. Exclude provider and model metadata from the predictive feature vector.
5. Retain eligible content metadata and historical performance fields for an auditable comparison with the capstone.

**Preprocessing**

- Numerical features: convert invalid values to missing and use median imputation.
- Categorical features: replace missing values with an explicit `unknown` category and use one-hot encoding.
- Unseen categorical values: handle safely without generating errors.
- Fit all preprocessing transformations on training data only during model evaluation.

**Temporal caveat**

The retained 90-day aggregate metrics may overlap the 30-day periods used to construct the target. They are therefore not certified as pre-outcome predictors. This feature vector supports an exploratory, observed-label classification exercise, not a validated future-decline forecast.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [1]:

from pathlib import Path
import subprocess
import sys
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# 1. Locate the repository and dataset.
REPO_URL = (
    "https://github.com/"
    "Kifayat-Sayed/flyrank-ml-internship1.git"
)

REPO_DIR = Path(
    "/content/flyrank-ml-internship1"
)

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
        check=True
    )

DATA_PATH = (
    REPO_DIR /
    "data/raw/content_refresh_anonymized.csv"
)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "The anonymized starter dataset was not found."
    )

# 2. Load the dataset.
data = pd.read_csv(DATA_PATH)

required_columns = {
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "impressions_90d",
    "days_since_last_update"
}

missing_columns = required_columns - set(data.columns)

assert not missing_columns, (
    f"Missing required columns: {missing_columns}"
)

# 3. Define the observed target.
data["is_declining_label"] = (
    data["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)

TARGET = "is_declining_label"

# 4. Match the capstone feature exclusions.
EXCLUDED = {
    "content_id",
    "client_id",
    TARGET,
    "trend_direction",
    "trend_pct",
    "impressions_last_30d",
    "impressions_prev_30d",
    "clicks_last_30d",
    "clicks_prev_30d",
    "sessions_last_30d",
    "sessions_prev_30d",
    "provider_used",
    "model_used"
}

feature_cols = [
    col for col in data.columns
    if col not in EXCLUDED
    and not data[col].isna().all()
]

X = data[feature_cols].copy()
y = data[TARGET].copy()
groups = data["client_id"].copy()

numeric_features = X.select_dtypes(
    include=["number", "bool"]
).columns.tolist()

categorical_features = [
    col for col in X.columns
    if col not in numeric_features
]

# 5. Build preprocessing pipelines.
numeric_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    )
])

categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="constant",
            fill_value="unknown"
        )
    ),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore"
        )
    )
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

print("FEATURE VECTOR CREATED")
print("=" * 45)
print("Dataset rows:", f"{len(data):,}")
print("Dataset columns:", len(data.columns))
print("Unique clients:", groups.nunique())
print("Predictive features:", len(feature_cols))
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Target prevalence:", f"{y.mean():.2%}")
print("Feature matrix shape:", X.shape)

print("\nFEATURE NAMES")
print(feature_cols)

assert TARGET not in feature_cols
assert "content_id" not in feature_cols
assert "client_id" not in feature_cols
assert "trend_direction" not in feature_cols
assert "trend_pct" not in feature_cols

print("\nDirect target-field exclusion checks: PASSED")


FEATURE VECTOR CREATED
Dataset rows: 30,000
Dataset columns: 45
Unique clients: 32
Predictive features: 32
Numeric features: 23
Categorical features: 9
Target prevalence: 54.21%
Feature matrix shape: (30000, 32)

FEATURE NAMES
['search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier']

Direct target-field exclusion checks: PASSED


## 3. The leakage hunt

### Feature definitions, missingness and availability

The feature vector contains numerical content-performance measurements and categorical content attributes.

**Numerical features**

- Search-volume and competition measurements describe keyword opportunity.
- Word count and character count describe content size.
- 90-day impressions, clicks, pageviews, sessions and engagement measurements describe observed activity.
- Content age and days since last update describe content freshness.
- CTR, average position and engagement rates describe observed search and user-engagement performance.

**Categorical features**

- `content_type` describes the content format.
- `main_intent` describes the recorded search intent.
- `competition_level` describes keyword competition.
- Age, freshness, word-count, impression and position tiers provide categorical representations of content attributes.

**Missing-value policy**

Numerical missing values are handled through median imputation, with missingness indicators where supported. Categorical missing values are represented by an explicit `unknown` category.

`avg_position = 0` is a special case: the data dictionary identifies zero as no position data, not a genuine search position. This requires separate interpretation and should not be treated as a literal rank.

**Availability assessment**

The supplied extract is a snapshot of trailing-90-day aggregates. It does not establish the exact reporting publication time for every individual feature.

Consequently, a feature being present in the dataset does not automatically establish that it was available before the prediction decision.

The 90-day performance features are retained to reproduce the capstone feature vector, but their overlap with the trend-label windows remains a temporal leakage concern.

In [2]:

# Build a feature-level audit table.
feature_notes = {
    "search_volume": "Keyword search-volume estimate",
    "competition": "Numerical keyword competition",
    "cpc": "Estimated cost per click",
    "word_count": "Content word count",
    "char_count": "Content character count",
    "content_age_days": "Age of content in days",
    "days_since_last_update": "Days since last update",
    "impressions_90d": "Search impressions over 90 days",
    "clicks_90d": "Search clicks over 90 days",
    "pageviews_90d": "Pageviews over 90 days",
    "sessions_90d": "Sessions over 90 days",
    "users_90d": "Users over 90 days",
    "engaged_sessions_90d": "Engaged sessions over 90 days",
    "ai_sessions_90d": "AI-referred sessions over 90 days",
    "scroll_events_90d": "Scroll events over 90 days",
    "days_with_impressions": "Days with search impressions",
    "days_with_sessions": "Days with sessions",
    "ctr": "Click-through rate",
    "avg_position": "Average search position",
    "engagement_rate": "Engagement rate",
    "scroll_rate": "Scroll-event rate",
    "ai_traffic_pct": "AI-referred traffic percentage",
    "competition_level": "Categorical competition level",
    "content_type": "Content format",
    "main_intent": "Recorded search intent",
    "age_tier": "Content-age category",
    "age_tier_order": "Ordered content-age category",
    "freshness_tier": "Content freshness category",
    "word_count_tier": "Word-count category",
    "char_count_tier": "Character-count category",
    "impression_tier": "Impression-volume category",
    "position_tier": "Search-position category"
}

# Features using trailing 90-day activity or rates derived from it.
performance_features = {
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "impression_tier",
    "position_tier"
}

audit_rows = []

for col in feature_cols:
    is_numeric = col in numeric_features

    if col in performance_features:
        availability = (
            "Snapshot timing and overlap with label window "
            "must be verified"
        )
    else:
        availability = (
            "Potentially available at review time; "
            "publication timing not independently verified"
        )

    if col == "avg_position":
        missing_policy = (
            "Zero requires special missing-data treatment; "
            "median imputation alone does not resolve zero sentinel"
        )
    elif is_numeric:
        missing_policy = (
            "Median imputation with missingness indicator"
        )
    else:
        missing_policy = (
            "Missing values represented as unknown"
        )

    audit_rows.append({
        "feature": col,
        "meaning": feature_notes.get(
            col, "Content attribute or derived category"
        ),
        "type": "Numerical" if is_numeric else "Categorical",
        "missing_percentage": round(
            data[col].isna().mean() * 100, 2
        ),
        "missing_handling": missing_policy,
        "temporal_assessment": availability
    })

feature_audit = pd.DataFrame(audit_rows)

print("FEATURE AVAILABILITY AND MISSINGNESS AUDIT")
display(feature_audit)

print("\nFEATURE TYPE SUMMARY")
print(feature_audit["type"].value_counts())

print("\nFEATURES WITH HIGHEST MISSINGNESS")
display(
    feature_audit.sort_values(
        "missing_percentage",
        ascending=False
    ).head(10)
)

print(
    "\nFeatures requiring temporal-window review:",
    len(performance_features.intersection(feature_cols))
)


FEATURE AVAILABILITY AND MISSINGNESS AUDIT


,feature,meaning,type,missing_percentage,missing_handling,temporal_assessment
0,search_volume,Keyword search-volume estimate,Numerical,8.23,Median imputation with missingness indicator,Potentially available at review time; publicat...
1,competition,Numerical keyword competition,Numerical,8.23,Median imputation with missingness indicator,Potentially available at review time; publicat...
2,competition_level,Categorical competition level,Categorical,8.70,Missing values represented as unknown,Potentially available at review time; publicat...
3,cpc,Estimated cost per click,Numerical,8.23,Median imputation with missingness indicator,Potentially available at review time; publicat...
4,content_type,Content format,Categorical,0.00,Missing values represented as unknown,Potentially available at review time; publicat...
5,main_intent,Recorded search intent,Categorical,7.91,Missing values represented as unknown,Potentially available at review time; publicat...
6,word_count,Content word count,Numerical,25.66,Median imputation with missingness indicator,Potentially available at review time; publicat...
7,char_count,Content character count,Numerical,25.66,Median imputation with missingness indicator,Potentially available at review time; publicat...
8,impressions_90d,Search impressions over 90 days,Numerical,0.00,Median imputation with missingness indicator,Snapshot timing and overlap with label window ...
9,clicks_90d,Search clicks over 90 days,Numerical,0.00,Median imputation with missingness indicator,Snapshot timing and overlap with label window ...



FEATURE TYPE SUMMARY
type
Numerical      23
Categorical     9
Name: count, dtype: int64

FEATURES WITH HIGHEST MISSINGNESS


,feature,meaning,type,missing_percentage,missing_handling,temporal_assessment
6,word_count,Content word count,Numerical,25.66,Median imputation with missingness indicator,Potentially available at review time; publicat...
7,char_count,Content character count,Numerical,25.66,Median imputation with missingness indicator,Potentially available at review time; publicat...
24,char_count_tier,Character-count category,Categorical,25.66,Missing values represented as unknown,Potentially available at review time; publicat...
23,word_count_tier,Word-count category,Categorical,25.66,Missing values represented as unknown,Potentially available at review time; publicat...
2,competition_level,Categorical competition level,Categorical,8.70,Missing values represented as unknown,Potentially available at review time; publicat...
1,competition,Numerical keyword competition,Numerical,8.23,Median imputation with missingness indicator,Potentially available at review time; publicat...
3,cpc,Estimated cost per click,Numerical,8.23,Median imputation with missingness indicator,Potentially available at review time; publicat...
0,search_volume,Keyword search-volume estimate,Numerical,8.23,Median imputation with missingness indicator,Potentially available at review time; publicat...
5,main_intent,Recorded search intent,Categorical,7.91,Missing values represented as unknown,Potentially available at review time; publicat...
28,scroll_rate,Scroll-event rate,Numerical,0.42,Median imputation with missingness indicator,Snapshot timing and overlap with label window ...



Features requiring temporal-window review: 17


## 4. What I excluded and why

### Adversarial leakage assessment

The target is derived from the observed trend direction, which itself is derived from the comparison between the most recent 30-day and preceding 30-day impressions.

Three leakage categories are examined.

**1. Direct label-derived leakage**

- `trend_direction` directly determines the target.
- `trend_pct` is used to derive the trend direction.
- Both fields must remain outside the predictive feature matrix.

**2. Temporal-window overlap**

The target is based on recent 30-day performance compared with the preceding 30 days.

The trailing-90-day aggregate metrics include periods that overlap these comparison windows. Therefore, excluding the direct label fields does not eliminate all temporal leakage risk.

The available snapshot does not provide sufficient evidence to certify these aggregate features as pre-outcome predictors.

**3. Decision-derived and identity leakage**

- `content_id` and `client_id` are identifiers, not predictive features.
- `provider_used` and `model_used` are excluded from the capstone feature vector.
- Client identifiers remain available separately for grouped validation.

**Deliberate leakage experiment**

A synthetic feature containing an exact copy of the target is introduced into a diagnostic model. Its purpose is to demonstrate how a label-derived feature can produce misleadingly perfect evaluation results.

This experiment is strictly a leakage demonstration. Its performance must never be reported as a legitimate model result.

In [3]:

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# 1. Verify direct label leakage exclusions.
direct_leakage_fields = {
    "trend_direction",
    "trend_pct",
    "is_declining_label"
}

identifier_fields = {
    "content_id",
    "client_id"
}

assert not (
    direct_leakage_fields.intersection(feature_cols)
), "Direct label leakage detected!"

assert not (
    identifier_fields.intersection(feature_cols)
), "Identifier leakage detected!"

print("DIRECT LEAKAGE CHECK: PASSED")
print("Target:", TARGET)
print("Excluded direct label fields:",
      sorted(direct_leakage_fields))
print("Excluded identifiers:",
      sorted(identifier_fields))

# 2. Verify the target definition.
expected_target = (
    data["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)

assert np.array_equal(
    data[TARGET].to_numpy(),
    expected_target.to_numpy()
)

print("\nTARGET DEFINITION CHECK: PASSED")

# 3. Identify temporal overlap risks.
temporal_risk_features = sorted(
    performance_features.intersection(feature_cols)
)

print("\nTEMPORAL WINDOW RISK")
print("Potentially overlapping performance features:",
      len(temporal_risk_features))

for feature in temporal_risk_features:
    print("-", feature)

# 4. Deliberately introduce a target-derived feature.
# This is a controlled leakage demonstration only.
demo = pd.DataFrame({
    "ordinary_feature": data["content_age_days"],
    "target_leak": data[TARGET],
    "target": data[TARGET]
}).dropna()

X_demo = demo[
    ["ordinary_feature", "target_leak"]
]

y_demo = demo["target"]

X_train_demo, X_test_demo, y_train_demo, y_test_demo = (
    train_test_split(
        X_demo,
        y_demo,
        test_size=0.20,
        random_state=42,
        stratify=y_demo
    )
)

leak_model = DecisionTreeClassifier(
    max_depth=1,
    random_state=42
)

leak_model.fit(
    X_train_demo,
    y_train_demo
)

leak_predictions = leak_model.predict(
    X_test_demo
)

leak_accuracy = accuracy_score(
    y_test_demo,
    leak_predictions
)

print("\nDELIBERATE LEAKAGE DEMONSTRATION")
print("Diagnostic test accuracy:",
      f"{leak_accuracy:.4f}")

assert leak_accuracy == 1.0, (
    "The deliberate target-leak demonstration "
    "did not produce perfect accuracy."
)

print(
    "Expected result: 100% accuracy because "
    "the feature directly contains the answer."
)

print("\nThis diagnostic result is NOT a valid model score.")
print("Temporal independence of the retained 90-day "
      "features remains unverified.")


DIRECT LEAKAGE CHECK: PASSED
Target: is_declining_label
Excluded direct label fields: ['is_declining_label', 'trend_direction', 'trend_pct']
Excluded identifiers: ['client_id', 'content_id']

TARGET DEFINITION CHECK: PASSED

TEMPORAL WINDOW RISK
Potentially overlapping performance features: 17
- ai_sessions_90d
- ai_traffic_pct
- avg_position
- clicks_90d
- ctr
- days_with_impressions
- days_with_sessions
- engaged_sessions_90d
- engagement_rate
- impression_tier
- impressions_90d
- pageviews_90d
- position_tier
- scroll_events_90d
- scroll_rate
- sessions_90d
- users_90d

DELIBERATE LEAKAGE DEMONSTRATION
Diagnostic test accuracy: 1.0000
Expected result: 100% accuracy because the feature directly contains the answer.

This diagnostic result is NOT a valid model score.
Temporal independence of the retained 90-day features remains unverified.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.